# RAG-Adapter research notebook
See `docs/notebooks.md` before running individual experiment sections. Outputs are cleared and local paths use the configuration cell below.


In [ ]:
from __future__ import annotations
import os
from pathlib import Path
RELEASE_ROOT = Path.cwd().resolve()
if RELEASE_ROOT.name == "notebooks":
    RELEASE_ROOT = RELEASE_ROOT.parent
DATA_ROOT = str(Path(os.environ.get("RAG_DATA_ROOT", RELEASE_ROOT / "data_local")).expanduser().resolve())
WORK_ROOT = str(Path(os.environ.get("RAG_WORK_ROOT", RELEASE_ROOT / "runs")).expanduser().resolve())
BGE_MODEL = os.environ.get("RAG_BGE_MODEL", "BAAI/bge-m3")
Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)


# 加载依赖

In [ ]:
import cv2
import random
import os
from time import strftime
from time import gmtime
import shutil
import re
from subprocess import call
from collections import defaultdict
import json


# 每秒1帧均匀采样

In [ ]:
# 对整个电影按照fps=1的帧率均匀采样
def get_video_frames(video_path, save_root):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if int(fps) == 0:
         return 
    frame_total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = int(frame_total / fps)

    video_name = os.path.basename(video_path).split(".")[0]
    save_folder = f"{save_root}/{video_name}"
#     print(save_folder)
    if os.path.exists(save_folder):
        # shutil.rmtree(save_folder)
        # os.makedirs(save_folder)
        return
    else:
        os.makedirs(save_folder)

    second = 1
    for i in range(duration):
            frame_num = int(i*fps)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = cap.read()
            save_pic_path = os.path.join(save_folder, f"{second:06d}.jpg")
            second += 1
            if not ret:
                print(f"Cannot read the frame: {video_path}")
                call(["ffmpeg", "-i", video_path, "-q:v", "1", "-vf", f"select=eq(n\,{frame_num})", "-vframes", "1", "-f", "image2", save_pic_path])
                continue
            cv2.imwrite(save_pic_path, frame)
    cap.release()
    # cv2.destroyAllWindows()
                


# 先判断是不是大于50s，然后在每秒一帧采样

In [ ]:
def get_video_frames_50s(video_path, save_root):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if int(fps) == 0:
         return 
    frame_total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = int(frame_total / fps)
    # 视频时长必须超过50s
    if duration <= 50 or duration > 300:
         return
    video_name = os.path.basename(video_path).split(".")[0]
    save_folder = f"{save_root}/{video_name}"
#     print(save_folder)
    if os.path.exists(save_folder):
        # shutil.rmtree(save_folder)
        # os.makedirs(save_folder)
        return
    else:
        os.makedirs(save_folder)

    second = 1
    for i in range(duration):
            frame_num = int(i*fps)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = cap.read()
            save_pic_path = os.path.join(save_folder, f"{second:06d}.jpg")
            second += 1
            if not ret:
                print(f"Cannot read the frame: {video_path}")
                call(["ffmpeg", "-i", video_path, "-q:v", "1", "-vf", f"select=eq(n\,{frame_num})", "-vframes", "1", "-f", "image2", save_pic_path])
                continue
            cv2.imwrite(save_pic_path, frame)
    cap.release()
                


# 对Video-MME数据集进行每秒一帧采样

In [ ]:
# 生成单个视频的frames
# movie_name = "0IdYJGBmguM.mp4"
# video_path = f"{DATA_ROOT}/dataset/Video-MME/videos_chunked_01/data/{movie_name}"
# get_video_frames(video_path)

# 一次性生成所有视频的frames
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/videos"):
    if len(ds) == 0:
        for f in fs: 
            if f.endswith(".mp4") or f.endswith(".mkv"):
                video_path = os.path.join(root, f)
                get_video_frames(video_path, f"{DATA_ROOT}/frames")


In [ ]:
# 生成采样的90个视频中的frames
import numpy as np

rsd = np.load(f"{WORK_ROOT}/data/rsd_final.npy", allow_pickle=True).item()
rmd = np.load(f"{WORK_ROOT}/data/rmd_final.npy", allow_pickle=True).item()
rld = np.load(f"{WORK_ROOT}/data/rld_final.npy", allow_pickle=True).item()

picked = []
for keys in rsd.keys():
    picked.extend(rsd[keys])
    picked.extend(rmd[keys])
    picked.extend(rld[keys])

for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/videos"):
    if len(ds) == 0:
        for f in fs: 
            name = f.split(".")[0]
            if name in picked:
                if f.endswith(".mp4") or f.endswith(".mkv"):
                    video_path = os.path.join(root, f)
                    get_video_frames(video_path)


## 测试生成指定Video-MME视频帧所需时间

In [ ]:
# 运行时间11.4s
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/videos"):
    if len(ds) == 0:
        for f in fs: 
            if f == "GrO5sxp3n0E.mp4":
                video_path = os.path.join(root, f)
                get_video_frames(video_path, f"{DATA_ROOT}/test")


# 对MLVU数据集每秒一帧采样

In [ ]:
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/added_video"):
    if len(ds) == 0:
        for f in fs: 
            if f.endswith(".mp4") or f.endswith(".mkv"):
                video_path = os.path.join(root, f)
                catagory = video_path.split("/")[-2]
                get_video_frames(video_path, f"{DATA_ROOT}/dataset/added_frames/{catagory}")


# 对 Percption Test Train set数据集每秒一帧采样

In [ ]:
with open(f"{DATA_ROOT}/dataset/Perception_Test/mc_question_train.json", "r") as json_file:
    mc_question = json.load(json_file)

# print(mc_question["video_9431"])

cnt = 0
# 采样90最长的视频
train_videos = dict()
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Perception_Test/videos"):
    if len(ds) == 0:
        for f in fs: 
            video_id = f.split(".mp4")[0]
            if video_id in mc_question:
                train_videos[video_id] = int(mc_question[video_id]["metadata"]["num_frames"])

sorted_videos = sorted(train_videos.items(), key=lambda x: x[1], reverse=True)
print(sorted_videos)

# for vid, num_frames in sorted_videos[:90]:
#     video_path = os.path.join("{DATA_ROOT}/dataset/Perception_Test/videos", f"{vid}.mp4")
#     target_path = os.path.join("{DATA_ROOT}/dataset/Perception_Test/sampled_videos", f"{vid}.mp4")
#     shutil.copy(video_path, target_path)
# print(sampled_videos)



In [ ]:
# 从90个视频中采样frames
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Perception_Test/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_path = os.path.join(root, f)
            get_video_frames(video_path, f"{DATA_ROOT}/dataset/Perception_Test/sampled_frames")


In [ ]:
# 构建问题代码
with open(f"{DATA_ROOT}/dataset/Perception_Test/mc_question_train.json", "r") as json_file:
    mc_question = json.load(json_file)

questions = defaultdict()
alts = ["A", "B", "C"]
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Perception_Test/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]
            if video_id not in questions:
                questions[video_id] = defaultdict(list)
            for q in mc_question[video_id]["mc_question"]:
                questions[video_id]["id"].append(q["id"])
                questions[video_id]["question"].append(q["question"])
                index_options = []
                for i, opt in enumerate(q["options"]):
                    index_options.append(f"{alts[i]}. {opt}")
                questions[video_id]["options"].append(index_options)
                questions[video_id]["answer_id"].append(alts[int(q["answer_id"])])

print(video_id)
for i in range(len(questions[video_id]["question"])):
    question = ""
    question += questions[video_id]["question"][i] + "\n"
    question += '\n'.join(questions[video_id]["options"][i]) + "\n"
    print(question)


# 测试视频帧相关性

In [ ]:
video_id = "1wzgMHrkrys"
q_id = "456-1"
yes = 0
no = 0
for root, ds, fs in os.walk(f"{DATA_ROOT}/results/Video-MME/chat-univi_select_by_itself/{video_id}/{q_id}"):
    if len(ds) == 0:
        for f in fs:
            file_path = os.path.join(root, f)
            with open(file_path, 'r', encoding='utf-8') as pred_file:
                text = pred_file.read()
            patterns = ["(yes)","(no)","(Yes)","(No)"]
            for p in patterns:
                ans = re.findall(p, text)
                if len(ans) != 0:
                    break
            if len(ans) == 0:
                print(text)
            else:
                if ans[0] in ["Yes", "yes"]:
                    yes += 1
                else:
                    no += 1
print(yes)
print(no)


# 对Egochame每秒一帧采样

In [ ]:
with open(f"{DATA_ROOT}/dataset/EgoSchema/questions.json", "r") as q_file:
    mc_questions = json.load(q_file)

with open(f"{DATA_ROOT}/dataset/EgoSchema/subset_answers.json", "r") as a_file:
    mc_answers = json.load(a_file)

qids = []
have_video_and_ansers_qids = []
have_answer_qids = mc_answers.keys()
for question in mc_questions:
    q_id = question["q_uid"]
    if q_id in have_answer_qids:
        qids.append(q_id)

have_videos = os.listdir(f"{DATA_ROOT}/dataset/EgoSchema/good_clips_git")
have_video_qids = [v.split(".")[0] for v in have_videos]
for qid in qids:
    if qid in have_video_qids:
        have_video_and_ansers_qids.append(qid)

# random.shuffle(have_video_and_ansers_qids)
# for qid in have_video_and_ansers_qids[:90]:
#     video_path = os.path.join("{DATA_ROOT}/dataset/EgoSchema/good_clips_git", f"{qid}.mp4")
#     target_path = os.path.join("{DATA_ROOT}/dataset/EgoSchema/sampled_videos", f"{qid}.mp4")
#     shutil.copy(video_path, target_path)


In [ ]:
# 运行4m0.9s
# 从90个视频中采样frames
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/EgoSchema/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_path = os.path.join(root, f)
            get_video_frames(video_path, f"{DATA_ROOT}/dataset/EgoSchema/sampled_frames")


In [ ]:
vid = os.listdir(f"{DATA_ROOT}/dataset/EgoSchema/sampled_videos")
vids = [v.split(".")[0] for v in vid]
sorted(vids)
for v in vids:
    print(v)


In [ ]:
# 构建问题代码
with open(f"{DATA_ROOT}/dataset/EgoSchema/questions.json", "r") as q_file:
    mc_questions = json.load(q_file)

with open(f"{DATA_ROOT}/dataset/EgoSchema/subset_answers.json", "r") as a_file:
    mc_answers = json.load(a_file)

questions = defaultdict()
alts = ["A", "B", "C", "D", "E"]
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/EgoSchema/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]
            if video_id not in questions:
                questions[video_id] = defaultdict(list)
            for video in mc_questions:

                if video["q_uid"] == video_id:
                    questions[video_id]["question"].append(video["question"])

                    index_options = []
                    index_options.append(f"A. {video['option 0']}")
                    index_options.append(f"B. {video['option 1']}")
                    index_options.append(f"C. {video['option 2']}")
                    index_options.append(f"D. {video['option 3']}")
                    index_options.append(f"E. {video['option 4']}")

                    questions[video_id]["options"].append(index_options)
                    questions[video_id]["answer_id"].append(alts[int(mc_answers[video_id])])
                
video_id = "0d173aa3-9a94-4ba4-84bc-949d3254a63d"
for i in range(len(questions[video_id]["question"])):
    question = ""
    question += questions[video_id]["question"][i] + "\n"
    question += '\n'.join(questions[video_id]["options"][i]) + "\n"
    print(question)


# 对MSVD-QA数据集进行每秒一帧采样

In [ ]:
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MSVD-QA/YouTubeClips"):
    if len(ds) == 0:
        for f in fs:
            # name = f.split(".")[0]
            video_path = os.path.join(root, f)
            get_video_frames(video_path, f"{DATA_ROOT}/dataset/MSVD-QA/frames")


# 对MSRVTT-QA数据集每秒采一帧

In [ ]:
# train and val
# for root, ds, fs in os.walk("{DATA_ROOT}/dataset/MSRVTT-QA/train_val_videos/TrainValVideo"):
#     if len(ds) == 0:
#         for f in fs: 
#             # name = f.split(".")[0]
#             video_path = os.path.join(root, f)
#             get_video_frames(video_path, "{DATA_ROOT}/dataset/MSRVTT-QA/frames")

# test
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MSRVTT-QA/test_videos/TestVideo"):
    if len(ds) == 0:
        for f in fs: 
            # name = f.split(".")[0]
            video_path = os.path.join(root, f)
            get_video_frames(video_path, f"{DATA_ROOT}/dataset/MSRVTT-QA/frames")


# 对ActivityNet-QA数据集每秒采帧

In [ ]:
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/ActivityNet-QA/ActivityNet_Test-1-3_videos/all_test"):
    if len(ds) == 0:
        for f in fs: 
            # name = f.split(".")[0]
            video_path = os.path.join(root, f)
            get_video_frames(video_path, f"{DATA_ROOT}/dataset/ActivityNet-QA/frames")

# get_video_frames("{DATA_ROOT}/dataset/ActivitNet-QA/activitynet_videos/v_1hB5jVAhSDE.mp4", "{DATA_ROOT}/dataset/ActivitNet-QA/frames")


# 对TGIF-QA数据集每秒采帧

In [ ]:
import imageio
from PIL import Image
import os                

def sample_frames_from_gif(gif_path, output_folder):
    """
    从 GIF 文件中每秒采样一帧图像并保存到指定文件夹。
    
    Args:
        gif_path (str): GIF 文件的路径。
        output_folder (str): 保存采样图像的文件夹路径。
    """

    # 使用 imageio 打开 GIF 文件
    try:
        gif = imageio.mimread(gif_path)
    except Exception as e:
        print(f"open Error: {gif_path}")
        return
    try:
        gif_duration = imageio.get_reader(gif_path).get_meta_data()['duration']  # 获取每帧持续时间 (ms)
        if gif_duration == 0:
            print(f"no duration: {gif_path}")
            return
    except Exception as e:
        print(f"no duration key: {gif_path}")
        return
    
    # 创建输出文件夹
    gif_name = os.path.basename(gif_path).split('.')[0]
    save_folder = os.path.join(output_folder, gif_name)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    else:
        return
    
    frame_rate = 1000 / gif_duration  # 计算帧率 (frames per second)

    # 按每秒一帧采样
    sample_interval = int(frame_rate)  # 每秒采样一帧
    if int(sample_interval) == 0:
        print(f"no frame_rate: {gif_path}")
        return
    index = 1
    for i in range(0, len(gif), sample_interval):
        frame = gif[i]
        # 将 numpy 数组转换为 PIL 图像
        frame_image = Image.fromarray(frame)
        # 保存图像
        frame_image.save(os.path.join(save_folder, f"{index:06d}.png"))
        index += 1

for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/TGIF-QA/gifs"):
    if len(ds) == 0:
        for f in fs: 
            if f.endswith(".gif"):
                gif_path = os.path.join(root, f)
                sample_frames_from_gif(gif_path, f"{DATA_ROOT}/dataset/TGIF-QA/frames")


# 对MMAT中所有大于50s的视频进行每秒采样1帧并进行标注

In [ ]:
# MSVD-QA
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MSVD-QA/YouTubeClips"):
    if len(ds) == 0:
        for f in fs: 
            # name = f.split(".")[0]
            video_path = os.path.join(root, f)
            get_video_frames_50s(video_path, f"{DATA_ROOT}/dataset/MMAT(50s)/MSVD-QA/frames")


In [ ]:
# MSRVTT-QA
# train and val
# for root, ds, fs in os.walk("{DATA_ROOT}/dataset/MSRVTT-QA/train_val_videos/TrainValVideo"):
#     if len(ds) == 0:
#         for f in fs: 
#             # name = f.split(".")[0]
#             video_path = os.path.join(root, f)
#             get_video_frames_50s(video_path, "{DATA_ROOT}/dataset/MMAT(50s)/MSRVTT-QA/frames")

# test
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MSRVTT-QA/test_videos/TestVideo"):
    if len(ds) == 0:
        for f in fs: 
            # name = f.split(".")[0]
            video_path = os.path.join(root, f)
            get_video_frames_50s(video_path, f"{DATA_ROOT}/dataset/MMAT(50s)/MSRVTT-QA/frames")


In [ ]:
# ActivityNet-QA
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/ActivityNet-QA/ActivityNet_Test-1-3_videos/all_test"):
    if len(ds) == 0:
        for f in fs: 
            # name = f.split(".")[0]
            video_path = os.path.join(root, f)
            get_video_frames_50s(video_path, f"{DATA_ROOT}/dataset/MMAT(50s)/ActivityNet-QA/frames")

# get_video_frames("{DATA_ROOT}/dataset/ActivitNet-QA/activitynet_videos/v_1hB5jVAhSDE.mp4", "{DATA_ROOT}/dataset/ActivitNet-QA/frames")


# 获取MMAT(50s)的测试视频

In [ ]:
# ActivityNet-QA (只保留test_q.json中的视频)
test_questions_dict = defaultdict(list)
json_path = f"{DATA_ROOT}/dataset/ActivityNet-QA/test_q.json"
file = open(json_path)
contents = [json.loads(line) for line in file][0]
for content in contents:
    video_id = "v_" + content["video_name"]
    question = content["question"]
    test_questions_dict[video_id].append(question)

video_ids = []
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MMAT(50s)/ActivityNet-QA/frames"):
    if len(ds) == 0:
        video_id = os.path.basename(root)
        # for f in fs: 
        if video_id in test_questions_dict:
            video_ids.append(video_id)
        else:
            shutil.rmtree(root)
print(len(video_ids))
            
